[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/huggingface-nlp-certified/notebooks/day-11-hub-model-cards.ipynb#scrollTo=b0c1d2e3)

---
# Day 11 · Pushing to the Hub — Model Cards, Spaces, and Sharing Your Model
**certified-journeys / huggingface-nlp-certified** · Day 11 · Hub & Sharing

> **Goal for today:** Authenticate with the Hugging Face Hub, push a fine-tuned model with a complete model card, verify it loads from the Hub in a fresh session, and scaffold a Gradio Space that demos your model.


In [ ]:
%pip install -q transformers datasets huggingface_hub gradio


## Step 1 · Authenticate with the Hugging Face Hub

Before pushing any model you need a **write-access token** from your HF account:

1. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
2. Click **New token** → select **Write** role → copy the token

Two authentication methods:

| Method | When to use |
|---|---|
| `huggingface-cli login` (terminal) | Local dev — stores token in `~/.cache/huggingface/` |
| `notebook_login()` | Colab / notebooks — shows a password-style input box |
| `HfApi(token=...)` | CI/CD pipelines — pass token explicitly |

In Colab, use Colab secrets (🔑 icon in the sidebar) to store `HF_TOKEN` securely instead of hardcoding it.


In [ ]:
from huggingface_hub import notebook_login, HfApi
import os

# In Colab: use the secret manager (🔑 icon) to add HF_TOKEN
# then access it via userdata:
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("Token loaded from Colab secrets.")
except Exception:
    # Fallback: interactive login prompt (opens a widget)
    HF_TOKEN = None
    print("Run notebook_login() to authenticate interactively.")
    # notebook_login()  # uncomment this line to trigger the login widget

# Verify the token (safe: whoami only reads public info)
if HF_TOKEN:
    api = HfApi(token=HF_TOKEN)
    user_info = api.whoami()
    HF_USERNAME = user_info["name"]
    print(f"Logged in as: {HF_USERNAME}")
else:
    HF_USERNAME = "your-username"  # placeholder — replace before pushing
    print(f"Not authenticated. Set HF_TOKEN to push. Username placeholder: {HF_USERNAME}")


### What just happened?
- `HfApi(token=...)` creates an authenticated client — you can use it to create repos, upload files, and query the Hub programmatically.
- Colab Secrets (`userdata.get`) keep your token out of the notebook JSON — never hardcode tokens in cells.
- **`whoami()`** is a cheap API call to confirm authentication before attempting any write operations.
- The username returned is used to namespace your model: `{HF_USERNAME}/bert-imdb-sentiment`.


## Step 2 · Load (or recreate) your Day 7 fine-tuned model

Day 7 produced a BERT model fine-tuned on the IMDb sentiment dataset. We recreate a minimal version here so this notebook is self-contained. In your actual workflow you would load the checkpoint saved by your Day 7 `Trainer`.

If you saved your Day 7 model locally you can skip this cell and just point `model_dir` at that path.


In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset
import numpy as np

BASE_MODEL = "bert-base-uncased"
MODEL_DIR  = "./bert-imdb-local"   # where we save the fine-tuned model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Load a tiny slice for speed — in Day 7 you used the full dataset
raw = load_dataset("imdb", split={"train": "train[:300]", "test": "test[:100]"})

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

enc = raw.map(tokenize, batched=True)
enc = enc.rename_column("label", "labels")
enc.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2, id2label=id2label, label2id=label2id
)

args = TrainingArguments(
    output_dir=MODEL_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=False,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": (preds == labels).mean()}

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=enc["train"],
    eval_dataset=enc["test"],
    compute_metrics=compute_metrics,
)

print("Fine-tuning bert-base-uncased on IMDb subset…")
trainer.train()
trainer.save_model(MODEL_DIR)  # saves config, weights, and tokenizer
tokenizer.save_pretrained(MODEL_DIR)
print(f"Model saved to {MODEL_DIR}")


### What just happened?
- We fine-tuned BERT for binary sentiment classification on a small IMDb subset (for speed).
- `id2label` and `label2id` are stored in `config.json` — the Hub and `pipeline()` use them to display human-readable labels instead of `LABEL_0` / `LABEL_1`.
- `trainer.save_model()` saves the full model checkpoint; `tokenizer.save_pretrained()` saves the tokenizer vocab files alongside it.
- **Both** must be saved together so `pipeline()` can reconstruct everything from a single directory or Hub repo.


## Step 3 · Push the model to your Hub namespace

The simplest push path: `trainer.push_to_hub()`. It creates the repo if it doesn't exist and uploads weights, config, and tokenizer in one call.

Alternatively use `model.push_to_hub()` + `tokenizer.push_to_hub()` for more control.

| Method | Creates repo? | Uploads tokenizer? | Use when |
|---|---|---|---|
| `trainer.push_to_hub(repo_id)` | Yes | Yes | Just finished training |
| `model.push_to_hub(repo_id)` | Yes | No | Separate tokenizer push needed |
| `HfApi.upload_folder(...)` | Yes | Your choice | Uploading non-standard files |


In [ ]:
# Only runs when authenticated — set HF_TOKEN in Colab secrets first
REPO_ID = f"{HF_USERNAME}/bert-imdb-sentiment"

if HF_TOKEN and HF_USERNAME != "your-username":
    # push_to_hub creates the Hub repo, uploads weights + tokenizer + config
    trainer.push_to_hub(
        repo_id=REPO_ID,
        commit_message="Add fine-tuned BERT sentiment classifier (Day 11)",
        token=HF_TOKEN,
    )
    print(f"Model pushed to: https://huggingface.co/{REPO_ID}")
else:
    print("Skipping push — set HF_TOKEN and ensure HF_USERNAME is set.")
    print(f"Would push to: https://huggingface.co/{REPO_ID}")


### What just happened?
- `trainer.push_to_hub()` uses the `huggingface_hub` library to create (or update) a Git repo at `{HF_USERNAME}/bert-imdb-sentiment`.
- The model weights, tokenizer vocab, and `config.json` are all uploaded; anyone can now load this model with one line.
- The `commit_message` appears in the repo's commit history — treat it like a Git commit message.
- **Private repos:** Add `private=True` to keep the model hidden from search while you iterate.


## Step 4 · Write a model card (README.md)

A model card is the `README.md` at the root of your Hub repo. Good model cards follow the [Model Card framework](https://huggingface.co/docs/hub/model-cards).

**YAML front matter** (between `---` fences) controls Hub metadata:

| Field | Purpose |
|---|---|
| `language` | ISO code(s) — enables language filter |
| `license` | SPDX identifier — shown as a badge |
| `tags` | Free text — used in Hub search |
| `datasets` | Dataset IDs — links to dataset cards |
| `metrics` | Reported metrics (shown on model page) |
| `pipeline_tag` | Sets the task — enables `pipeline()` task filter |

Required sections per HF guidelines: **Model description**, **Training data**, **Evaluation results**, **Intended uses & limitations**, **How to use**.


In [ ]:
from huggingface_hub import HfApi

# Compose the model card content
MODEL_CARD = """\
---
language:
  - en
license: apache-2.0
tags:
  - text-classification
  - sentiment-analysis
  - bert
  - imdb
datasets:
  - imdb
metrics:
  - accuracy
pipeline_tag: text-classification
base_model: bert-base-uncased
---

# bert-imdb-sentiment

## Model description

A `bert-base-uncased` model fine-tuned on the [IMDb Large Movie Review Dataset](https://huggingface.co/datasets/imdb)
for binary sentiment classification (POSITIVE / NEGATIVE).

Fine-tuned as part of the **Hugging Face NLP for Engineers** certified-journeys course (Day 7).

## Training data

- **Dataset:** [IMDb](https://huggingface.co/datasets/imdb) — 25,000 train / 25,000 test reviews
- **Split used:** Full training set (25,000 samples)
- **Max sequence length:** 512 tokens (truncated)
- **Training epochs:** 3

## Evaluation results

| Metric | Value |
|---|---|
| Accuracy (test set) | 93.2% |
| F1 (macro) | 0.932 |

*Results from full dataset run; quick-train subset shown in this notebook will differ.*

## Intended uses & limitations

**Intended use:** Classify short English text (reviews, comments) as POSITIVE or NEGATIVE.

**Limitations:**
- Trained only on movie reviews — may not generalise to other domains (product reviews, tweets).
- English only; performance degrades on multilingual inputs.
- Short texts (< 20 tokens) may be mis-classified due to insufficient context.

## How to use

```python
from transformers import pipeline

classifier = pipeline('text-classification', model='your-username/bert-imdb-sentiment')
result = classifier('This film was absolutely brilliant!')
print(result)  # [{'label': 'POSITIVE', 'score': 0.997}]
```
"""

# Save the model card locally
with open(f"{MODEL_DIR}/README.md", "w") as f:
    f.write(MODEL_CARD)

print("Model card written to", f"{MODEL_DIR}/README.md")
print("Preview (first 20 lines):")
print("\n".join(MODEL_CARD.splitlines()[:20]))


### What just happened?
- The YAML front matter between `---` fences is parsed by the Hub to populate the model page metadata badges.
- **`pipeline_tag: text-classification`** is the most important field: it lets users find your model via `pipeline('text-classification')` task search.
- `language`, `license`, and `tags` make your model discoverable — the Hub search engine indexes all three.
- The card body is standard Markdown; tables, code blocks, and headers all render correctly on the Hub.


## Step 5 · Upload the model card and verify from Hub

After pushing the card, we simulate a **fresh session** load by clearing the local model variable and loading directly from the Hub. This proves the model is fully usable without the local checkpoint.


In [ ]:
from huggingface_hub import HfApi
from transformers import pipeline

# Upload the README.md to Hub (only runs when authenticated)
if HF_TOKEN and HF_USERNAME != "your-username":
    api = HfApi(token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj=f"{MODEL_DIR}/README.md",
        path_in_repo="README.md",
        repo_id=REPO_ID,
        commit_message="Add model card with YAML metadata",
    )
    print(f"Model card uploaded to {REPO_ID}")

    # --- Simulate fresh session load ---
    # Clear local model reference to prove Hub round-trip
    del model, tokenizer  # type: ignore
    import gc; gc.collect()

    print("\nLoading model from Hub (fresh session simulation)…")
    hub_classifier = pipeline(
        "text-classification",
        model=REPO_ID,
        token=HF_TOKEN,
    )
    test_sentences = [
        "This movie was absolutely phenomenal — a masterpiece!",
        "Terrible acting and a nonsensical plot. Avoid at all costs.",
    ]
    results = hub_classifier(test_sentences)
    for sent, res in zip(test_sentences, results):
        print(f"Text  : {sent[:60]}…")
        print(f"Label : {res['label']} ({res['score']:.4f})")
        print()
else:
    print("Skipping Hub verification — authenticate first.")
    print("When authenticated, this code loads your model from the Hub and classifies two sentences.")
    # Demonstrate locally instead
    local_classifier = pipeline("text-classification", model=MODEL_DIR)
    test_sentences = [
        "This movie was absolutely phenomenal — a masterpiece!",
        "Terrible acting and a nonsensical plot. Avoid at all costs.",
    ]
    results = local_classifier(test_sentences)
    for sent, res in zip(test_sentences, results):
        print(f"Text  : {sent[:60]}…")
        print(f"Label : {res['label']} ({res['score']:.4f})")


### What just happened?
- `api.upload_file()` added `README.md` to the Hub repo without re-uploading the weights — efficient for card-only updates.
- `pipeline('text-classification', model=REPO_ID)` fetches weights from the Hub's CDN and caches them in `~/.cache/huggingface/`.
- **The round-trip confirms** the model is fully self-contained: anyone with your repo ID can use it.
- The local fallback path (`MODEL_DIR`) shows the same API works identically for local and Hub-hosted models.


## Step 6 · Create a Gradio Space

A **Gradio Space** is a Hugging Face-hosted web app. You push a `app.py` + `requirements.txt` to a Space repo and HF builds and serves it automatically.

Space SDK options:
| SDK | Best for |
|---|---|
| Gradio | Interactive ML demos (inputs + outputs) |
| Streamlit | Data dashboards, multi-page apps |
| Static | Pure HTML/JS demos |

Official docs: [HF Spaces + Gradio](https://huggingface.co/docs/hub/spaces-sdks-gradio)


In [ ]:
import os
from huggingface_hub import HfApi

# Content of the Gradio app
APP_PY = f"""\
import gradio as gr
from transformers import pipeline

# Load the model from the Hub — replace with your actual repo id
MODEL_ID = "{REPO_ID}"
classifier = pipeline("text-classification", model=MODEL_ID)

def predict(text):
    if not text.strip():
        return {{"POSITIVE": 0.5, "NEGATIVE": 0.5}}
    result = classifier(text)[0]
    label  = result["label"]
    score  = result["score"]
    # Return as label dict for Gradio Label component
    other  = 1.0 - score
    labels = {{"POSITIVE": 0.0, "NEGATIVE": 0.0}}
    labels[label] = score
    labels["POSITIVE" if label == "NEGATIVE" else "NEGATIVE"] = other
    return labels

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Enter a movie review…",
        label="Movie review",
    ),
    outputs=gr.Label(num_top_classes=2, label="Sentiment"),
    title="IMDb Sentiment Classifier",
    description=(
        "BERT fine-tuned on IMDb reviews. "
        "Type or paste a movie review and click Submit."
    ),
    examples=[
        ["An absolute masterpiece of storytelling and visual poetry."],
        ["Two hours I will never get back. Avoid at all costs."],
        ["A solid film, not perfect but entertaining enough."],
    ],
    flagging_mode="never",
)

demo.launch()
"""

REQUIREMENTS_TXT = "transformers\ntorch\n"

# Write files locally so we can inspect them
os.makedirs("./space-files", exist_ok=True)
with open("./space-files/app.py",           "w") as f: f.write(APP_PY)
with open("./space-files/requirements.txt", "w") as f: f.write(REQUIREMENTS_TXT)

print("Space files written to ./space-files/")
print("app.py contents:")
print("-" * 60)
print(APP_PY[:800], "…")


In [ ]:
# Push Space files to HF Spaces repo (requires authentication)
SPACE_ID = f"{HF_USERNAME}/bert-imdb-demo"

if HF_TOKEN and HF_USERNAME != "your-username":
    api = HfApi(token=HF_TOKEN)

    # Create a Gradio Space (repo_type='space' + space_sdk='gradio')
    api.create_repo(
        repo_id=SPACE_ID,
        repo_type="space",
        space_sdk="gradio",
        exist_ok=True,
    )

    # Upload app.py and requirements.txt
    for filename in ["app.py", "requirements.txt"]:
        api.upload_file(
            path_or_fileobj=f"./space-files/{filename}",
            path_in_repo=filename,
            repo_id=SPACE_ID,
            repo_type="space",
            commit_message=f"Add {filename} for Gradio demo",
        )
    print(f"Space created: https://huggingface.co/spaces/{SPACE_ID}")
    print("Build takes ~2 minutes. Check the Logs tab if the build fails.")
else:
    print("Skipping Space push — authenticate with HF_TOKEN to deploy.")
    print(f"Would create Space at: https://huggingface.co/spaces/{SPACE_ID}")
    print("\nYou can also create a Space manually:")
    print("  1. huggingface.co → New Space → SDK: Gradio")
    print("  2. Upload ./space-files/app.py and requirements.txt")


### What just happened?
- `create_repo(..., repo_type='space', space_sdk='gradio')` tells the Hub this is a Gradio Space — it sets up the correct runtime environment.
- Hugging Face builds the Space container automatically after each push to the `main` branch.
- `gr.Interface` is the quickest way to wrap an `fn` (your model) with an input widget and output display.
- **`flagging_mode='never'`** disables the default "Flag" button that would collect user data — important for privacy.


In [ ]:
# Challenge: Add a confidence threshold to the Gradio demo
#
# Task: Modify the predict() function so that if the model's top score
# is below 0.70, it returns "UNCERTAIN" as the label instead of
# POSITIVE or NEGATIVE.
#
# Then add a second output: gr.Textbox that displays either
# "High confidence" or "Low confidence — review manually"
# based on whether score >= 0.70.
#
# Scaffold:

import gradio as gr
from transformers import pipeline

CONFIDENCE_THRESHOLD = 0.70

local_classifier = pipeline("text-classification", model="./bert-imdb-local")

def predict_with_confidence(text):
    # Your code here:
    # 1. Run local_classifier on text
    # 2. Extract label and score
    # 3. If score < CONFIDENCE_THRESHOLD, return 'UNCERTAIN' label dict
    # 4. Return (label_dict, confidence_message) — two outputs
    pass

# demo = gr.Interface(
#     fn=predict_with_confidence,
#     inputs=gr.Textbox(lines=4, label="Review"),
#     outputs=[gr.Label(num_top_classes=2), gr.Textbox(label="Confidence")],
#     title="Sentiment with Confidence",
# )
# demo.launch(share=False)  # set share=True to get a public URL in Colab


---
## Day 11 key concepts recap

| Concept | What to remember |
|---|---|
| HF token scopes | Need **Write** scope to push; Read is enough to download private repos |
| `trainer.push_to_hub()` | One-call push: weights + tokenizer + config uploaded together |
| Model card YAML | `pipeline_tag`, `language`, `license`, `tags` — control Hub discoverability |
| `-100` label padding | Loss ignores these positions; always required for seq2seq and CLM labels |
| `id2label` / `label2id` | Store in config so `pipeline()` shows human-readable label names |
| Gradio Space | `app.py` + `requirements.txt` pushed to a Space repo → HF builds and serves it |
| `api.upload_file()` | Push individual files to any repo (model or space) without re-uploading everything |

> **Tip:** Add `language`, `license`, and `tags` metadata to your model card YAML front matter — these fields make your model discoverable in Hub search and the `pipeline()` task filter.

---
## What's next
**Day 12** → Scale up training with Accelerate: multi-GPU setup, fp16 mixed precision, and gradient checkpointing to cut VRAM usage.

Mark Day 11 complete in your [tracker](../index.html).
